### Scenario 3: Multiple data scientists working on multiple ML models
MLflow setup:

- Tracking server: yes, remote server (EC2).
- Backend store: postgresql database.
- Artifacts store: s3 bucket.
- The experiments can be explored by accessing the remote server.

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file mlflow_on_aws.md to create a new AWS account and launch the tracking server.

### READ

- In my case, I'm simulating AWS services with local containers, 3 containers, one for MLFLOW server, POSTGRES DATABASE and S3 bucket wit MINIO
- local host 9001 -> minio
- local host 5000 -> mlflow server
- Postgres is not exposed to localhost
- See mlflow-local-stack folder to see configuration of this simulation

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

### Conectar al servidor MLflow , in this case local running into a container

In [2]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin123"

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")
mlflow.search_experiments()

tracking URI: 'http://localhost:5000'


[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1778732292445, experiment_id='0', last_update_time=1778732292445, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### registering runs

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/05/13 23:29:00 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/05/13 23:29:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 23:29:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: 'mlflow-artifacts:/1/4bc81fdc733c4c2f9ec8e0657c066454/artifacts'
🏃 View run intrigued-fawn-488 at: http://localhost:5000/#/experiments/1/runs/4bc81fdc733c4c2f9ec8e0657c066454
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778732940958, experiment_id='1', last_update_time=1778732940958, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1778732292445, experiment_id='0', last_update_time=1778732292445, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://localhost:5000")

In [7]:
client.search_registered_models()

[]

In [8]:
run_id = client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/05/13 23:31:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 4bc81fdc733c4c2f9ec8e0657c066454 has no artifacts at artifact path 'models', registering model based on models:/m-f41e1c0bb17740d9b0f171c0ba3030d4 instead
2026/05/13 23:31:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1778733060210, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1778733060210, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='4bc81fdc733c4c2f9ec8e0657c066454', run_link='', source='models:/m-f41e1c0bb17740d9b0f171c0ba3030d4', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>